In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(_root, "tools")) and os.path.isdir(os.path.join(_root, "data")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from tools.paths import data_path, outputs_path

In [ ]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────────────────────────────────────
placementhistory = pd.read_csv(data_path('CareerLaunch2026PlacementHistory.csv'), index_col=False)

print("=== Shape & columns ===")
print(placementhistory.shape)
print(placementhistory.columns.tolist())

print("\n=== Dtypes (raw) ===")
print(placementhistory.dtypes)

# Clean up types
placementhistory['Changed Date'] = pd.to_datetime(placementhistory['Changed Date'], format='%m/%d/%Y')
placementhistory['Student Code'] = placementhistory['Student Code'].astype('Int64')  # nullable int, drops the ".0"

print("\n=== Opportunity Status value counts ===")
print(placementhistory['Opportunity Status'].value_counts())

print("\n=== Hub ('Group Name') value counts ===")
print(placementhistory['Group Name'].value_counts())

print("\n=== Placement Status value counts (raw log rows, NOT deduped) ===")
print(placementhistory['Placement Status'].value_counts())

print("\n=== Date range covered ===")
print(placementhistory['Changed Date'].min(), '->', placementhistory['Changed Date'].max())

print("\n=== Daily change volume (helps spot bulk-match days) ===")
print(placementhistory['Changed Date'].value_counts().sort_index())

# ─────────────────────────────────────────────────────────────────────────────
# DROP DUPLICATES, KEEP LATEST RECORD
# ─────────────────────────────────────────────────────────────────────────────
matched = placementhistory.dropna(subset=['Student Code']).copy()
dupe_counts = matched.groupby(['Opportunity Id', 'Student Code']).size()
print(f"\n{(dupe_counts > 1).sum()} of {len(dupe_counts)} opportunity-student pairs have more than one log row")

# Take the LATEST status per opportunity-student pair (sort by date, keep last)
matched_sorted = matched.sort_values(['Opportunity Id', 'Student Code', 'Changed Date', 'Changed Time'])
latest_status = matched_sorted.groupby(['Opportunity Id', 'Student Code']).tail(1).copy()

print(f"\n=== Deduped to {len(latest_status)} unique opportunity-student final records ===")
print(latest_status['Placement Status'].value_counts())

confirmed = latest_status[latest_status['Placement Status'] == 'Confirmed'].copy()

# ─────────────────────────────────────────────────────────────────────────────
# ONE-STUDENT-ONE-PLACEMENT RULE
# ─────────────────────────────────────────────────────────────────────────────
pre_deduped_count = len(confirmed)
confirmed = confirmed.sort_values(['Student Code', 'Changed Date', 'Changed Time'])
confirmed = confirmed.groupby('Student Code').tail(1).copy()

print(f"\n=== One-student-one-placement rule ===")
print(f"Confirmed records before dedup: {pre_deduped_count}")
print(f"Confirmed records after keeping latest per student: {len(confirmed)}")
print(f"Superseded (earlier, replaced) records dropped: {pre_deduped_count - len(confirmed)}")

# ─────────────────────────────────────────────────────────────────────────────
# CORE MATCHING FUNNEL (final status only)
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== Final status by hub ===")
funnel = latest_status.groupby(['Group Name', 'Placement Status']).size().unstack(fill_value=0)
print(funnel)

print(f"\nTotal CONFIRMED placements: {len(confirmed)}")
print(f"Unique students confirmed: {confirmed['Student Code'].nunique()}")
print(f"Unique agencies matched:   {confirmed['Agency Id'].nunique()}")

print("\n=== Confirmed matches & unique agencies by hub ===")
by_hub = confirmed.groupby('Group Name').agg(
    students_matched=('Student Code', 'nunique'),
    agencies_matched=('Agency Id', 'nunique'),
    opportunities_used=('Opportunity Id', 'nunique'),
)
print(by_hub)

# ─────────────────────────────────────────────────────────────────────────────
# BULK MATCHING COUNTS
# ─────────────────────────────────────────────────────────────────────────────
BULK_ACCOUNT = 'elijahnunez@gmail.com'
BULK_START_DATE = pd.Timestamp('2026-06-09')  # EN's bulk match run began here

print("\n=== Created By value counts, top 15 (staff/bulk accounts vs. student/coordinator IDs) ===")
print(confirmed['Created By'].value_counts().head(15))


def classify_match_type(row):
    if BULK_ACCOUNT in row['Created By']:
        return 'Bulk Match'
    if row['Changed Date'] < BULK_START_DATE:
        return 'Pre-Match'
    return 'Manual Match'


confirmed['match_type'] = confirmed.apply(classify_match_type, axis=1)

print("\n=== Match type totals (overall) ===")
print(confirmed['match_type'].value_counts())

print("\n=== Match type by hub ===")
match_type_by_hub = confirmed.groupby(['Group Name', 'match_type']).size().unstack(fill_value=0)
print(match_type_by_hub)

print("\n=== Final summary table ===")
summary = confirmed.groupby('Group Name').agg(
    Unique_Students_Matched=('Student Code', 'nunique'),
    Unique_Agencies_Matched=('Agency Id', 'nunique'),
).join(
    confirmed.pivot_table(index='Group Name', columns='match_type', values='Student Code',
                           aggfunc='nunique', fill_value=0)
)
print(summary)
print(f"\nTotal Unique Agencies Matched (all hubs): {confirmed['Agency Id'].nunique()}")


In [ ]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# LOAD & SCOPE — same header-offset fix and CL26 ground-truth scoping used
# throughout (see the Data Validation / QA section at the end of this
# notebook for the full explanation).
# ─────────────────────────────────────────────────────────────────────────────
apps = pd.read_csv(data_path('CL26StudentOpportunityApplications (2).csv'), index_col=False)
ph = pd.read_csv(data_path('CareerLaunch2026PlacementHistory.csv'), index_col=False)

cl26_opps = ph[['Opportunity Id', 'Group Name', 'Agency Id', 'Agency Name']].drop_duplicates('Opportunity Id')
scoped = apps.merge(cl26_opps, on='Opportunity Id', how='inner')

print(f"Total scoped applications: {len(scoped)}")
print(f"Total unique agencies applied to: {scoped['Agency Name'].nunique()}")

# ─────────────────────────────────────────────────────────────────────────────
# TOP 10 AGENCIES BY HUB — raw counts alone are naive here, since hubs have
# very different total application volumes (5,118 in CSS vs. 7,606 in
# Marketing). Reporting each agency's applications as a % of ITS OWN hub's
# total makes the numbers comparable across hubs and shows how concentrated
# (or spread out) student interest actually is within each one.
# ─────────────────────────────────────────────────────────────────────────────
hubs = [
    '2026 Career Launch - Community and Social Services',
    '2026 Career Launch - Healthcare',
    '2026 Career Launch - Marketing and Communications',
    '2026 Career Launch - STEM and Green',
]

top10_by_hub = {}

for hub in hubs:
    sub = scoped[scoped['Group Name'] == hub]
    hub_total = len(sub)

    counts = sub.groupby('Agency Name').size().sort_values(ascending=False)
    top10 = counts.head(10).to_frame('Applications')
    top10['Pct of Hub Applications'] = (100 * top10['Applications'] / hub_total).round(1)

    top10_by_hub[hub] = top10

    print(f"\n=== {hub} (hub total applications = {hub_total}) ===")
    print(top10.to_string())

# top10_by_hub['2026 Career Launch - Healthcare'] etc. -- ready to feed into
# the docx build script as (name, count, pct) tuples per hub.

In [ ]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# LOAD — same header/data mismatch as the placement history file (7 header
# names, 8 fields per row due to a trailing per-student sequence counter).
# index_col=False keeps the real 7 columns aligned and drops the counter.
# ─────────────────────────────────────────────────────────────────────────────
apps = pd.read_csv(data_path('CL26StudentOpportunityApplications.csv'), index_col=False)

print("=== Raw shape & columns ===")
print(apps.shape)
print(apps.columns.tolist())

# ─────────────────────────────────────────────────────────────────────────────
# SCOPE TO CL26 — the full/unfiltered export (pulled without a hub filter in
# InPlace) mixes in: other program years, other campaigns, and internal
# QA/test entries (e.g. Student Code "000000"/"11111111", comments like
# "Cultural Corps staff testing InPlace features. Disregard application.").
#
# Rather than pattern-matching junk directly (fragile), we scope to CL26 by
# keeping only rows whose Opportunity Id appears in this cycle's placement
# history -- that file is ground truth for "real CL26 2026 opportunity,
# any of the four hubs." Anything else is out of scope by definition.
# ─────────────────────────────────────────────────────────────────────────────
ph = pd.read_csv(data_path('CareerLaunch2026PlacementHistory.csv'), index_col=False)
cl26_opportunities = ph[['Opportunity Id', 'Opportunity Name', 'Group Name', 'Agency Id', 'Agency Name']].drop_duplicates('Opportunity Id')

print(f"\nCL26 opportunities (ground truth, all 4 hubs): {cl26_opportunities['Opportunity Id'].nunique()}")
print(f"Applications rows before scoping: {len(apps)}")

apps_scoped = apps.merge(cl26_opportunities, on='Opportunity Id', how='inner')

print(f"Applications rows after scoping to CL26 opportunities: {len(apps_scoped)}")
print(f"Dropped as out-of-scope (other programs/years/test entries): {len(apps) - len(apps_scoped)}")

print("\n=== Scoped applications by hub ===")
print(apps_scoped['Group Name'].value_counts())

# ─────────────────────────────────────────────────────────────────────────────
# BACKSTOP FLAGS — junk that happens to share an Opportunity Id with a real
# CL26 opportunity wouldn't get caught by the scoping step above. Flag (don't
# silently drop) anything that still looks suspicious so it can be reviewed.
# ─────────────────────────────────────────────────────────────────────────────
JUNK_COMMENT_KEYWORDS = ['disregard application', 'will be withdrawn', 'staff testing inplace']
suspicious_comment = apps_scoped['Opportunity Application Comments'].str.lower().str.contains(
    '|'.join(JUNK_COMMENT_KEYWORDS), na=False
)

# Repeated-digit codes like 000000, 11111111 read as very low-entropy strings
student_code_str = apps_scoped['Student Code'].astype(str)
suspicious_code = student_code_str.str.match(r'^(\d)\1*$', na=False)

flagged = apps_scoped[suspicious_comment | suspicious_code]
print(f"\n=== Flagged for manual review (suspicious comment or repeated-digit code): {len(flagged)} rows ===")
if len(flagged):
    print(flagged[['Student Code', 'Opportunity Id', 'Opportunity Application Status', 'Opportunity Application Comments']].head(10))

apps_clean = apps_scoped[~(suspicious_comment | suspicious_code)].copy()
print(f"\nFinal clean, CL26-scoped applications: {len(apps_clean)}")

# ─────────────────────────────────────────────────────────────────────────────
# CORE METRICS ON THE CLEANED DATA
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== Application status breakdown ===")
print(apps_clean['Opportunity Application Status'].value_counts())

print("\n=== Unique students who applied ===")
print(apps_clean['Student Code'].nunique())

print("\n=== Applications by hub ===")
print(apps_clean['Group Name'].value_counts())

print("\n=== Coordinator Preference Rank coverage ===")
print(f"{apps_clean['Coordinator Preference Rank'].notna().sum()} of {len(apps_clean)} rows have a coordinator rank")

# Students who applied vs. who ended up confirmed (needs placement history dedup)
matched = ph.dropna(subset=['Student Code']).copy()
matched['Changed Date'] = pd.to_datetime(matched['Changed Date'], format='%m/%d/%Y')
matched['Student Code'] = matched['Student Code'].astype('Int64')
matched_sorted = matched.sort_values(['Opportunity Id', 'Student Code', 'Changed Date', 'Changed Time'])
latest = matched_sorted.groupby(['Opportunity Id', 'Student Code']).tail(1)
confirmed = latest[latest['Placement Status'] == 'Confirmed'].copy()
confirmed = confirmed.sort_values(['Student Code', 'Changed Date', 'Changed Time']).groupby('Student Code').tail(1)
confirmed_students = set(confirmed['Student Code'].astype(int).unique())

apps_clean['Student Code'] = apps_clean['Student Code'].astype(int)
applicant_students = set(apps_clean['Student Code'].unique())

print(f"\n=== Applicant funnel ===")
print(f"Students who applied (clean, CL26-scoped): {len(applicant_students)}")
print(f"Of those, confirmed placement:             {len(applicant_students & confirmed_students)}")
print(f"Of those, NOT confirmed:                   {len(applicant_students - confirmed_students)}")

# Split the non-confirmed group: still pending ("Applied" only) vs resolved with no path forward
non_confirmed = apps_clean[apps_clean['Student Code'].isin(applicant_students - confirmed_students)]
status_by_student = non_confirmed.groupby('Student Code')['Opportunity Application Status'].apply(set)
still_pending = status_by_student[status_by_student.apply(lambda s: s == {'Applied'})]
resolved_no_placement = status_by_student[~status_by_student.index.isin(still_pending.index)]

print(f"\nNon-confirmed, still pending (Applied only, no resolution): {len(still_pending)}")
print(f"Non-confirmed, resolved with no placement (Declined/Unsuccessful mix): {len(resolved_no_placement)}")

## Data Validation / QA

Folded in from the standalone `validate_application_data.ipynb` — scoping/dedup logic
duplicates the cells above, but this section adds two diagnostics not covered elsewhere:
what specifically gets dropped by CL26 scoping (not just how many rows), and whether any
confirmed placement is missing a corresponding application record. Uses the
`CL26StudentOpportunityApplications (2).csv` export (no BACKSTOP junk-filtering applied,
unlike the cell above) — kept as-is rather than reconciled with the other cells' junk-filtered
version, since that's a real difference in what each was checking, not an accident.

In [ ]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# LOAD — same header/data mismatch as every other InPlace export (7 header
# names, 8 fields per row due to a trailing per-student sequence counter).
# index_col=False keeps the real 7 columns aligned and drops the counter.
# ─────────────────────────────────────────────────────────────────────────────
apps = pd.read_csv(data_path('CL26StudentOpportunityApplications (2).csv'), index_col=False)
ph = pd.read_csv(data_path('CareerLaunch2026PlacementHistory.csv'), index_col=False)

print("=== Raw applications file ===")
print("Total rows:", len(apps))
print("Unique students:", apps['Student Code'].nunique())
print("Unique opportunities:", apps['Opportunity Id'].nunique())

# ─────────────────────────────────────────────────────────────────────────────
# SCOPE TO CL26 — placement history is ground truth for "real CL26 2026
# opportunity, any of the four hubs." Anything else (other programs/years,
# test entries) gets dropped by the inner join, regardless of how the
# applications export itself was filtered upstream in InPlace.
# ─────────────────────────────────────────────────────────────────────────────
cl26_opps = ph[['Opportunity Id', 'Group Name', 'Agency Id', 'Agency Name']].drop_duplicates('Opportunity Id')
scoped = apps.merge(cl26_opps, on='Opportunity Id', how='inner')

print(f"\nRows after scoping to CL26 ground-truth opportunities: {len(scoped)}")
print(f"Dropped as out-of-scope: {len(apps) - len(scoped)}")
print(f"Unique students after scoping: {scoped['Student Code'].nunique()}")

print("\n=== Scoped applications by hub ===")
print(scoped['Group Name'].value_counts(dropna=False))

print("\n=== Application status breakdown (scoped) ===")
print(scoped['Opportunity Application Status'].value_counts())

# ─────────────────────────────────────────────────────────────────────────────
# WHAT GOT DROPPED — sanity check that the scoping step is still doing real
# work even on an export that claims to already be filtered to 2026.
# ─────────────────────────────────────────────────────────────────────────────
opp_not_in_ph = set(apps['Opportunity Id']) - set(cl26_opps['Opportunity Id'])
orphans = apps[apps['Opportunity Id'].isin(opp_not_in_ph)]
print(f"\n=== Out-of-scope rows (Opportunity Id not in CL26 placement history): {len(orphans)} ===")
print(f"Unique students in those orphan rows: {orphans['Student Code'].nunique()}")
print(orphans['Opportunity Application Status'].value_counts())

# ─────────────────────────────────────────────────────────────────────────────
# CANONICAL FINAL PLACEMENTS — same one-student-one-placement pipeline used
# throughout, needed to check the applicant funnel below.
# ─────────────────────────────────────────────────────────────────────────────
ph['Changed Date'] = pd.to_datetime(ph['Changed Date'], format='%m/%d/%Y')
ph['Student Code'] = ph['Student Code'].astype('Int64')
matched = ph.dropna(subset=['Student Code']).copy()
matched_sorted = matched.sort_values(['Opportunity Id', 'Student Code', 'Changed Date', 'Changed Time'])
latest = matched_sorted.groupby(['Opportunity Id', 'Student Code']).tail(1).copy()
confirmed = latest[latest['Placement Status'] == 'Confirmed'].copy()
confirmed = confirmed.sort_values(['Student Code', 'Changed Date', 'Changed Time']).groupby('Student Code').tail(1).copy()
confirmed_students = set(confirmed['Student Code'].astype(int).unique())

applicant_students = set(scoped['Student Code'].unique())

print("\n=== Applicant funnel ===")
print(f"Students who applied (clean, CL26-scoped): {len(applicant_students)}")
print(f"Of those, confirmed placement:             {len(applicant_students & confirmed_students)}")
print(f"Of those, NOT confirmed:                   {len(applicant_students - confirmed_students)}")
print(f"Confirmed students missing an application record: {len(confirmed_students - applicant_students)}")